<a href="https://colab.research.google.com/github/SanjanaDhawale/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanjanaDhawale/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
# ML-04 Setup

!pip -q install duckdb huggingface_hub pyarrow pandas

import os
import duckdb
import pandas as pd
from google.colab import userdata

# Read the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Set it as an environment variable
os.environ["HF_TOKEN"] = HF_TOKEN

print("✅ HF_TOKEN loaded successfully.")
print("DuckDB version:", duckdb.__version__)

✅ HF_TOKEN loaded successfully.
DuckDB version: 1.3.2


In [16]:
from huggingface_hub import whoami
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print(whoami(token=token))

{'type': 'user', 'id': '69d544d8d08ed3bdba647d16', 'name': 'SanjanaDhawale', 'fullname': 'Sanjana Dhawale', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/dTzyrMtQ19STU_i3vGavf.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'FlyRank Internship', 'role': 'fineGrained', 'createdAt': '2026-07-24T08:43:11.896Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '69d544d8d08ed3bdba647d16', 'type': 'user', 'name': 'SanjanaDhawale'}, 'permissions': ['repo.content.read']}]}}}}


In [17]:
from google.colab import userdata
import duckdb

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
);
""")

print("✅ Hugging Face secret configured in DuckDB")

✅ Hugging Face secret configured in DuckDB


In [18]:
con.execute("""
SELECT COUNT(*)
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
LIMIT 1;
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,78835655


In [19]:
con.execute("""
DESCRIBE
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
LIMIT 1;
""").fetchdf()


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [20]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet';
"""

con.execute(query).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,78835655,78829265


In [21]:
query = """
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet';
"""

con.execute(query).fetchdf()

,first_date,last_date
0,2025-01-27,2026-06-30


In [22]:
query = """
SELECT DISTINCT month
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
ORDER BY month;
"""

con.execute(query).fetchdf()

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


In [23]:
query = """
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
GROUP BY ga4_data_available;
"""

con.execute(query).fetchdf()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows
0,False,46383873
1,<NA>,29635327
2,True,2816455


## Unit of Analysis

One row represents the daily performance metrics of a single content item (`content_hash_id`) for a specific client (`client_hash_id`) on a particular `report_date`.

## Time Window

The dataset time window is verified using the minimum and maximum values of `report_date` from the warehouse.

In [24]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet';
"""

con.execute(query).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows,first_date,last_date
0,78835655,78829265,2025-01-27,2026-06-30


## Features

- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events

## Label

No label is defined in this notebook. This data contract identifies candidate features for downstream content opportunity scoring.

## Context

- report_date
- client_hash_id
- content_hash_id
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

## Excluded

Rows where `ga4_data_available` is FALSE or NULL are excluded when using GA4 engagement metrics because the required engagement data is unavailable.

In [25]:
query = """
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
GROUP BY ga4_data_available
ORDER BY ga4_data_available;
"""

con.execute(query).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows
0,False,46383873
1,True,2816455
2,<NA>,29635327


## Verification

The data contract was verified using SQL queries to confirm:

- Dataset grain (one row per report_date, client_hash_id, and content_hash_id)
- Total row count
- Time window using MIN(report_date) and MAX(report_date)
- Available monthly partitions
- GA4 data availability
- Dataset schema using DESCRIBE

In [26]:
# Verify grain, counts, missing values, and time window

query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS missing_ga4_flag
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet';
"""

con.execute(query).fetchdf()

print("\nGA4 availability:")
con.execute("""
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
GROUP BY ga4_data_available
ORDER BY ga4_data_available;
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


GA4 availability:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows
0,False,46383873
1,True,2816455
2,<NA>,29635327


## Data Limits

- Many rows do not contain GA4 engagement data.
- Rows where `ga4_data_available` is FALSE or NULL cannot be used for GA4-based analyses.
- Some historical periods contain only GSC metrics.
- The warehouse contains multiple monthly partitions, so analyses should use a consistent time window.
- This dataset contains anonymized identifiers (`client_hash_id` and `content_hash_id`) rather than human-readable values.

In [27]:
query = """
SELECT
    month,
    COUNT(*) AS rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
GROUP BY month
ORDER BY month;
"""

con.execute(query).fetchdf()

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.